In [1]:
import pandas as pd
from pandasql import sqldf 
import numpy as np

In [2]:
df = pd.read_csv('steam_games.csv', low_memory=False)

In [3]:

result = sqldf("select * from df where price_status ='paid' order by price DESC limit 10 ")

abv = sqldf("select developers ,avg(price) from df group by developers order by avg(price) DESC limit 10")

In [4]:
result.columns

Index(['app_id', 'name', 'release_date', 'price', 'price_status',
       'estimated_owners', 'developers', 'publishers', 'genres', 'categories',
       'tags', 'positive', 'negative', 'recommendations', 'peak_ccu',
       'metacritic_score', 'user_score', 'average_playtime_forever',
       'median_playtime_forever', 'average_playtime_2weeks',
       'median_playtime_2weeks', 'short_description', 'about_the_game',
       'detailed_description', 'notes', 'achievements', 'dlc_count',
       'packages', 'supported_languages', 'full_audio_languages', 'windows',
       'mac', 'linux', 'header_image', 'screenshots', 'movies', 'website',
       'support_url', 'support_email', 'metacritic_url',
       'steam_store_available', 'steam_spy_available'],
      dtype='str')

In [5]:
#DELETING ALL UNNECCESRY DATA

In [6]:
drop_cols = [
    'app_id',
    'header_image',
    'screenshots',
    'movies',
    'website',
    'support_url',
    'support_email',
    'metacritic_url',
    'steam_store_available',
    'steam_spy_available',
    'notes',
    'packages',
    'detailed_description',
    'recommendations','about_the_game'
    # keeping short_description, it tells a LOT about the genre
]

df = df.drop(columns=drop_cols, errors='ignore')

In [7]:
df.columns

Index(['name', 'release_date', 'price', 'price_status', 'estimated_owners',
       'developers', 'publishers', 'genres', 'categories', 'tags', 'positive',
       'negative', 'peak_ccu', 'metacritic_score', 'user_score',
       'average_playtime_forever', 'median_playtime_forever',
       'average_playtime_2weeks', 'median_playtime_2weeks',
       'short_description', 'achievements', 'dlc_count', 'supported_languages',
       'full_audio_languages', 'windows', 'mac', 'linux'],
      dtype='str')

In [8]:
df_staging = sqldf("""
    SELECT
        lower(trim(name))                             AS name,
        substr(release_date, 6, 2)    AS release_month,
        substr(release_date, 1, 4)    AS release_year,
        price,
        lower(trim(price_status))                     AS price_status,
        lower(trim(estimated_owners))                 AS estimated_owners,
        lower(trim(developers))                       AS developers,
        lower(trim(publishers))                       AS publishers,
        lower(trim(genres))                           AS genres,
        lower(trim(categories))                       AS categories,
        lower(trim(tags))                             AS tags,
        trim(short_description)                       AS short_description,
        positive,
        negative,
        peak_ccu,
        metacritic_score,
        user_score,
        average_playtime_forever,
        median_playtime_forever,
        average_playtime_2weeks,
        median_playtime_2weeks,
        achievements,
        dlc_count,
        lower(trim(supported_languages))              AS supported_languages,
        lower(trim(full_audio_languages))             AS full_audio_languages,
        windows,
        mac,
        linux
    FROM df
""")

In [9]:
#we have estimated_owners as bounderies we should take the middle point 

In [10]:
def owners_mid(s):
    if pd.isna(s):
        return np.nan
    try:
        s = str(s).replace(',', '')        # kill commas
        lo, hi = s.replace('..', '-').split('-')   # unify separator
        return (int(lo.strip()) + int(hi.strip())) / 2
    except (ValueError, AttributeError):
        return np.nan

df_staging['owners_mid'] = df_staging['estimated_owners'].apply(owners_mid)

In [11]:
sqldf("""
    select *
    from df_staging
    where owners_mid is null
      and estimated_owners is not null
""")

,name,release_month,release_year,price,price_status,estimated_owners,developers,publishers,genres,categories,...,average_playtime_2weeks,median_playtime_2weeks,achievements,dlc_count,supported_languages,full_audio_languages,windows,mac,linux,owners_mid


In [12]:
df_staging.drop(columns='estimated_owners',inplace=True)

In [13]:
df_staging.isnull().mean().sort_values(ascending=False)

metacritic_score            0.276143
full_audio_languages        0.220282
achievements                0.120175
short_description           0.059679
price                       0.008397
release_year                0.000595
release_month               0.000595
supported_languages         0.000136
name                        0.000115
mac                         0.000107
linux                       0.000107
windows                     0.000107
positive                    0.000000
tags                        0.000000
categories                  0.000000
genres                      0.000000
developers                  0.000000
publishers                  0.000000
price_status                0.000000
negative                    0.000000
median_playtime_2weeks      0.000000
average_playtime_2weeks     0.000000
median_playtime_forever     0.000000
average_playtime_forever    0.000000
user_score                  0.000000
peak_ccu                    0.000000
dlc_count                   0.000000
o

select_dtypes Return a subset of the DataFrame's columns based on the column dtypes.

In [14]:
# what fraction is NaN
print("NaN fraction:")
print(df_staging.isna().mean().sort_values(ascending=False).head(15))

# what fraction is zero (numeric only)
print("\nZero fraction:")
print((df_staging.select_dtypes('number') == 0).mean().sort_values(ascending=False))

NaN fraction:
metacritic_score        0.276143
full_audio_languages    0.220282
achievements            0.120175
short_description       0.059679
price                   0.008397
release_year            0.000595
release_month           0.000595
supported_languages     0.000136
name                    0.000115
mac                     0.000107
linux                   0.000107
windows                 0.000107
positive                0.000000
tags                    0.000000
categories              0.000000
dtype: float64

Zero fraction:
user_score                  0.999713
average_playtime_2weeks     0.983751
median_playtime_2weeks      0.983751
average_playtime_forever    0.915303
median_playtime_forever     0.915303
linux                       0.871894
peak_ccu                    0.858748
dlc_count                   0.856032
mac                         0.829890
metacritic_score            0.693129
negative                    0.516234
positive                    0.423621
achievements    

In [15]:
# how many games have a real (>0) user_score
print("games with user_score > 0:", (df_staging['user_score'] > 0).sum())
print("total games:", len(df_staging))

# check the distribution of nonzero scores
print(df_staging.loc[df_staging['user_score'] > 0, 'user_score'].describe())

games with user_score > 0: 40
total games: 139580
count     40.000000
mean      75.250000
std       17.642824
min       37.000000
25%       62.500000
50%       77.000000
75%       89.000000
max      100.000000
Name: user_score, dtype: float64


In [16]:
import numpy as np

# "no score" → -1
df_staging['metacritic_score'] = df_staging['metacritic_score'].fillna(-1)

# "no achievements" → 0
df_staging['achievements'] = df_staging['achievements'].fillna(0)

# "empty list" → ''
df_staging['full_audio_languages'] = df_staging['full_audio_languages'].fillna('')
df_staging['supported_languages']  = df_staging['supported_languages'].fillna('')
df_staging['short_description']    = df_staging['short_description'].fillna('')

# "truly unknown numeric" → median
df_staging['price'] = df_staging['price'].fillna(df_staging['price'].median())

# near-zero missingness → just drop those rows
df_staging = df_staging.dropna(subset=['name', 'windows', 'mac', 'linux', 'release_month'])

In [17]:
df_staging = df_staging.drop(columns=['user_score']) # TOO FEW INFO

In [18]:
df_staging['genres'] 

0                ["adventure", "casual", "indie", "sports"]
1         ["action", "adventure", "indie", "simulation",...
2         ["casual", "free to play", "indie", "early acc...
3                             ["adventure", "indie", "rpg"]
4                                       ["action", "indie"]
                                ...                        
139575               [{"id": "9", "description": "racing"}]
139576    [{"id": "1", "description": "action"}, {"id": ...
139577             [{"id": "2", "description": "strategy"}]
139578    [{"id": "1", "description": "action"}, {"id": ...
139579    [{"id": "1", "description": "action"}, {"id": ...
Name: genres, Length: 139496, dtype: str

In [19]:
import ast

def to_list(s):
    if pd.isna(s):
        return []
    # already a list?
    if isinstance(s, list):
        items = s
    else:
        # try JSON / literal-eval parse
        try:
            items = ast.literal_eval(s)  
        except (ValueError, SyntaxError):
            # fallback: split on ';'
            items = [x.strip() for x in str(s).split(';')]

    out = []
    for item in items:
        if isinstance(item, dict):
            # take the 'description' field
            v = item.get('description') or item.get('name')
            if v:
                out.append(str(v).strip().lower())
        else:
            out.append(str(item).strip().lower())
    return out

for c in ['genres', 'categories', 'tags']:
    df_staging[c] = df_staging[c].apply(to_list)

In [20]:
for c in ['genres', 'categories', 'tags']:
    bad = df_staging[c].apply(lambda x: not isinstance(x, list) or
                                        any(not isinstance(i, str) for i in x))
    print(c, "bad rows:", bad.sum())
    if bad.sum():
        print(df_staging.loc[bad, c].head())

genres bad rows: 0
categories bad rows: 0
tags bad rows: 0


In [21]:


# unique set
unique_genres = sorted({g for lst in df_staging['genres'] for g in lst})
print(unique_genres)
print("total unique:", len(unique_genres))



['360 video', 'accounting', 'action', 'adventure', 'animation & modeling', 'audio production', 'aventura', 'aventure', 'casual', 'design & illustration', 'documentary', 'early access', 'education', 'episodic', 'free to play', 'game development', 'gore', 'gratuit', 'indie', 'indépendant', 'massively multiplayer', 'movie', 'nudity', 'occasionnel', 'photo editing', 'racing', 'rpg', 'sexual content', 'short', 'simuladores', 'simulation', 'software training', 'sports', 'strategie', 'strategy', 'stratégie', 'tutorial', 'utilities', 'video production', 'violent', 'web publishing', 'インディー', '早期アクセス']
total unique: 43


In [22]:
RENAME = {
    'indépendant': 'indie',
    'インディー': 'indie',
    'strategie': 'strategy',
    'stratégie': 'strategy',
    'occasionnel': 'casual',
    'aventura': 'adventure',
    'aventure': 'adventure',
    'simuladores': 'simulation',
    '早期アクセス': 'early access',
    'gratuit': 'free to play',
    'nudity': 'adult content',
    'sexual content': 'adult content',

    # categories come in a bunch of languages too, same thing different name
    'achiziții în aplicație': 'in-app purchases',
    'caméra et confort de vue': 'camera comfort',
    'comodidad de la cámara': 'camera comfort',
    'compat. contrôleurs complète': 'full controller support',
    'controles de volumen personalizados': 'custom volume controls',
    'contrôle du volume différencié': 'custom volume controls',
    'cooperativ': 'co-op',
    'cooperativos': 'co-op',
    '協力プレイ': 'co-op',
    'cooperativ online': 'online co-op',
    'cooperativos en línea': 'online co-op',
    'オンライン協力プレイ': 'online co-op',
    'difficulté ajustable': 'adjustable difficulty',
    'jouable sans saisie en temps imparti': 'playable without timed input',
    'se puede jugar sin eventos de acción rápida': 'playable without timed input',
    'logros de steam': 'steam achievements',
    'succès steam': 'steam achievements',
    'mai mulți jucători': 'multi-player',
    'multijugador': 'multi-player',
    'マルチプレイヤー': 'multi-player',
    'option clavier uniquement': 'keyboard only option',
    'option souris uniquement': 'mouse only option',
    'option tactile uniquement': 'touch only option',
    'partage familial': 'family sharing',
    'préstamo familiar': 'family sharing',
    'ファミリーシェアリング': 'family sharing',
    'pvp online': 'online pvp',
    'solo': 'single-player',
    'un jucător': 'single-player',
    'un jugador': 'single-player',
    'シングルプレイヤー': 'single-player',
    'sonido estéreo': 'stereo sound',
    'vr supported': 'vr support',
}

def fix(tags):
    if not isinstance(tags, list):
        return []
    return sorted({RENAME.get(t, t) for t in tags})

In [23]:
for col in ['genres', 'categories', 'tags']:
    df_staging[col] = df_staging[col].apply(fix)

In [24]:
# how many languages the game supports, a 1 language game and a 25 language game are not the same kind of game
# the column looks like ["english, russian"] so we split on the commas
df_staging['n_languages'] = df_staging['supported_languages'].apply(
    lambda s: len([l for part in to_list(s) for l in part.split(',') if l.strip()])
)

In [25]:
df_staging_1 = df_staging.copy()

In [26]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

mlb_genres = MultiLabelBinarizer()
mlb_categories = MultiLabelBinarizer()
mlb_tags = MultiLabelBinarizer()

X_genres = pd.DataFrame(
    mlb_genres.fit_transform(df_staging_1['genres']),
    columns=[f'genres:{c}' for c in mlb_genres.classes_],
    index=df_staging_1.index,
)

X_categories = pd.DataFrame(
    mlb_categories.fit_transform(df_staging_1['categories']),
    columns=[f'categories:{c}' for c in mlb_categories.classes_],
    index=df_staging_1.index,
)

X_tags = pd.DataFrame(
    mlb_tags.fit_transform(df_staging_1['tags']),
    columns=[f'tags:{c}' for c in mlb_tags.classes_],
    index=df_staging_1.index,
)

df_staging_1 = df_staging_1.drop(columns=['genres', 'categories', 'tags'])
df_staging_1 = pd.concat([df_staging_1, X_genres, X_categories, X_tags], axis=1)

In [27]:
df_staging_1

,name,release_month,release_year,price,price_status,developers,publishers,short_description,positive,negative,...,tags:web publishing,tags:well-written,tags:werewolves,tags:western,tags:wholesome,tags:word game,tags:world war i,tags:world war ii,tags:wrestling,tags:zombies
0,big pogo man,06,2018,4.99,paid,"[""woojudev""]","[""woojudev""]","Climb mountain made of odds and ends, riding P...",48,12,...,0,0,0,0,0,0,0,0,0,0
1,hyperspace delivery service,06,2019,9.99,paid,"[""zotnip""]","[""zotnip""]",You’ve been hired to make a delivery to planet...,42,4,...,0,0,0,0,0,0,0,0,0,0
2,vr slots 3d,08,2018,0.00,free,"[""belka vr""]","[""belka vr""]",Try for free the first VR Slots in the world. ...,17,18,...,0,0,0,0,0,0,0,0,0,0
3,underwater hunting,06,2018,0.99,paid,"[""gibar""]","[""robot""]",Underwater hunting: Plunge into the underwater...,40,33,...,0,0,0,0,0,0,0,0,0,0
4,greenspawn restaurant,07,2018,3.99,paid,"[""apple dash""]","[""apple dash""]",Greenspawn Restaurant is a unique combination ...,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139575,race - the wtcc game,11,2006,39.99,paid,"[""simbin""]","[""simbin""]",The Official WTCC Game | RACE is the first eve...,121,42,...,0,0,0,0,0,0,0,0,0,0
139576,s.t.a.l.k.e.r.: shadow of chernobyl,03,2007,19.99,paid,"[""gsc game world""]","[""gsc game world""]",S.T.A.L.K.E.R.: Shadow of Chernobyl tells a st...,44761,2530,...,0,0,0,0,0,0,0,0,0,0
139577,medieval ii: total war™ kingdoms,08,2007,24.99,paid,"[""creative assembly"", ""feral interactive (mac)...","[""sega"", ""feral interactive (mac)"", ""feral int...",Medieval II: Total War Kingdoms is the officia...,599,31,...,0,0,0,0,0,0,0,0,0,0
139578,natural selection 2,10,2012,4.99,paid,"[""unknown worlds entertainment""]","[""unknown worlds entertainment"", ""krafton, inc.""]",A fast paced multiplayer shooter that pits ali...,8827,1539,...,0,0,0,0,0,0,0,0,0,0


In [28]:
import ast

def first(x):
    parsed = ast.literal_eval(x)
    return parsed[0] if parsed else 'unknown'

for col in ['developers', 'publishers']:
    df_staging_1[col] = df_staging_1[col].apply(first)

In [29]:
df_staging_1

,name,release_month,release_year,price,price_status,developers,publishers,short_description,positive,negative,...,tags:web publishing,tags:well-written,tags:werewolves,tags:western,tags:wholesome,tags:word game,tags:world war i,tags:world war ii,tags:wrestling,tags:zombies
0,big pogo man,06,2018,4.99,paid,woojudev,woojudev,"Climb mountain made of odds and ends, riding P...",48,12,...,0,0,0,0,0,0,0,0,0,0
1,hyperspace delivery service,06,2019,9.99,paid,zotnip,zotnip,You’ve been hired to make a delivery to planet...,42,4,...,0,0,0,0,0,0,0,0,0,0
2,vr slots 3d,08,2018,0.00,free,belka vr,belka vr,Try for free the first VR Slots in the world. ...,17,18,...,0,0,0,0,0,0,0,0,0,0
3,underwater hunting,06,2018,0.99,paid,gibar,robot,Underwater hunting: Plunge into the underwater...,40,33,...,0,0,0,0,0,0,0,0,0,0
4,greenspawn restaurant,07,2018,3.99,paid,apple dash,apple dash,Greenspawn Restaurant is a unique combination ...,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139575,race - the wtcc game,11,2006,39.99,paid,simbin,simbin,The Official WTCC Game | RACE is the first eve...,121,42,...,0,0,0,0,0,0,0,0,0,0
139576,s.t.a.l.k.e.r.: shadow of chernobyl,03,2007,19.99,paid,gsc game world,gsc game world,S.T.A.L.K.E.R.: Shadow of Chernobyl tells a st...,44761,2530,...,0,0,0,0,0,0,0,0,0,0
139577,medieval ii: total war™ kingdoms,08,2007,24.99,paid,creative assembly,sega,Medieval II: Total War Kingdoms is the officia...,599,31,...,0,0,0,0,0,0,0,0,0,0
139578,natural selection 2,10,2012,4.99,paid,unknown worlds entertainment,unknown worlds entertainment,A fast paced multiplayer shooter that pits ali...,8827,1539,...,0,0,0,0,0,0,0,0,0,0


In [30]:
df_staging_1['publishers'].unique() #71414 also too little compared to our data 139k we drop
df_staging_1.drop(columns='publishers',inplace=True)

In [31]:
df_staging_1['developers'].unique() #len 79046 too much too little information (overfitting possibility)
df_staging_1.drop(columns='developers',inplace=True)

In [32]:
df_staging_1.loc[df_staging_1['name'].str.len().sort_values(ascending=False).index].head(20)

,name,release_month,release_year,price,price_status,short_description,positive,negative,peak_ccu,metacritic_score,...,tags:web publishing,tags:well-written,tags:werewolves,tags:western,tags:wholesome,tags:word game,tags:world war i,tags:world war ii,tags:wrestling,tags:zombies
4304,萌盟大冒险 cute adventure 可爱 射击 生存 shoote...,11,2019,6.99,paid,Interesting island adventure，Go online with fr...,4,3,0,0.0,...,0,0,0,0,0,0,0,0,0,0
57130,shattered dreams: that one time when all my ap...,09,2024,4.99,paid,"Do you hate your life? Well, boy, do I have th...",14,6,0,0.0,...,0,0,0,0,0,0,0,0,0,0
118296,the epic and amazing adventure of gleepglorrpg...,07,2026,0.99,unavailable,play as gleepglorrpgilliandoohiggintilliousMMD...,0,0,0,-1.0,...,0,0,0,0,0,0,0,0,0,0
105053,sniper camouflage orange spy warrior i buried ...,08,2026,9.99,unavailable,Sniper simulator game with sponge bob graphics,0,0,0,-1.0,...,0,0,0,0,0,0,0,0,0,0
42205,mon-yu: defeat monsters and gain strong weapon...,09,2023,49.99,paid,Defeat the Devil Kings! You have been guided t...,9,4,0,0.0,...,0,0,0,0,0,0,0,0,0,0
87231,"i was a failed day trader who lost everything,...",08,2026,2.99,unavailable,A failed day trader dies at 3:47 a.m. and wake...,0,0,0,-1.0,...,0,0,0,0,0,0,0,0,0,0
67170,"for a long time, i built plans to steal a boy ...",09,2024,2.99,paid,"Watching my attractive colleague from work, I ...",12,4,0,0.0,...,0,0,0,0,0,0,0,0,0,0
56725,a 20-year-old rpg maker game remade with the l...,03,2025,0.00,free,The game that was created 20 years ago with RP...,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
125342,dvd screensaver simulator rebirth reimagined d...,07,2026,0.00,free,Experience the ultimate re-imagining of the cl...,0,0,0,-1.0,...,0,0,0,0,0,0,0,0,0,0
16489,vr time travelling in medieval towns and islan...,08,2020,9.99,paid,Medieval Western European towns emerged in the...,1,1,0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [33]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
model.save("models/all-MiniLM-L6-v2")   # app.py loads the encoder from here

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
import torch
print(torch.cuda.is_available())
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

True


In [35]:
names = df_staging_1['name'].fillna('').astype(str).tolist()

In [36]:
X_name = model.encode(
    names,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   
)

Batches:   0%|          | 0/545 [00:00<?, ?it/s]

In [37]:
# same thing for the short description, empty ones just become the "empty text" vector
X_desc = model.encode(
    df_staging_1['short_description'].astype(str).tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(X_name.shape, X_desc.shape)   # (139496, 384) (139496, 384)
print(X_name.dtype)   # float32

Batches:   0%|          | 0/545 [00:00<?, ?it/s]

(139496, 384) (139496, 384)
float32


In [38]:
df_staging_2 = df_staging_1.drop(columns=['name', 'short_description'])

In [39]:
X_name_df = pd.DataFrame(
    X_name,
    columns=[f'emb_{i}' for i in range(X_name.shape[1])],
    index=df_staging_2.index,
)
X_desc_df = pd.DataFrame(
    X_desc,
    columns=[f'desc_emb_{i}' for i in range(X_desc.shape[1])],
    index=df_staging_2.index,
)

df_staging_2 = pd.concat([X_name_df, X_desc_df, df_staging_2], axis=1)

In [40]:
print(df_staging_2.isna().sum().sum())

0


In [41]:
df_staging_2

,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,...,tags:web publishing,tags:well-written,tags:werewolves,tags:western,tags:wholesome,tags:word game,tags:world war i,tags:world war ii,tags:wrestling,tags:zombies
0,-0.001515,0.104078,0.019943,-0.036421,-0.020613,-0.088352,0.129516,-0.021646,-0.011214,0.038553,...,0,0,0,0,0,0,0,0,0,0
1,-0.023669,-0.032390,0.014752,-0.022427,-0.052468,-0.025907,0.035089,-0.048289,-0.054690,0.074642,...,0,0,0,0,0,0,0,0,0,0
2,-0.014476,-0.068003,-0.085496,-0.051485,-0.107899,-0.030715,-0.038414,-0.032351,0.037624,-0.004114,...,0,0,0,0,0,0,0,0,0,0
3,-0.036225,0.020533,0.009683,0.027999,0.026765,-0.101996,0.047210,-0.059294,-0.059315,0.026437,...,0,0,0,0,0,0,0,0,0,0
4,0.009698,0.032273,-0.005428,-0.024815,-0.053965,0.022666,0.018367,-0.043421,0.015694,-0.038415,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139575,-0.044019,0.084680,-0.021334,0.015307,0.009482,0.054987,0.008888,0.007841,-0.011464,0.023415,...,0,0,0,0,0,0,0,0,0,0
139576,-0.011139,0.071625,0.027137,0.077682,0.031027,-0.029798,0.018963,0.065146,-0.009167,0.000667,...,0,0,0,0,0,0,0,0,0,0
139577,-0.016374,0.065088,-0.009301,-0.029288,-0.097187,-0.030758,-0.103008,-0.064896,-0.016777,0.022663,...,0,0,0,0,0,0,0,0,0,0
139578,-0.012394,-0.006744,0.075919,0.071170,0.130215,0.038256,0.037376,-0.067846,-0.040551,0.097031,...,0,0,0,0,0,0,0,0,0,0


In [42]:
from pathlib import Path

out = Path('processed')
out.mkdir(parents=True, exist_ok=True)

# only parquet, the csv version was 800MB+ and nothing reads it
df_staging_2.to_parquet(out / 'df_staging_2.parquet', index=False)